# Hudson Link Performance, Oct 16–28 2026

One section per metric, read from the tables `process.ipynb` writes to `data/processed/` (run that first). The window is forced to **service days Oct 16–28**: before Oct 16 a recorder bug left days nearly empty, and Oct 29 is cut off partway through.

In [ ]:
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter, PercentFormatter

from shared.plots.style import apply_theme, add_source_footnote

apply_theme()

START, END = "20261016", "20261028"  # GTFS service days (start_date), inclusive
PROCESSED = Path("data/processed")
CHARTS = Path("outputs/2026-10-16_2026-10-28/charts")
CHARTS.mkdir(parents=True, exist_ok=True)
SOURCE = "Hudson Link real-time feed (CoachUSA) recorded via gtfs.zone; schedule from 511NY GTFS"
GREEN, GRAY, BLUE = "#3D9970", "#BDBDBD", "#2A6F97"


def load(name):
    return pd.read_parquet(PROCESSED / name / "data.parquet")


def in_window(df):
    return df[df["start_date"].between(START, END)].copy()


positions = in_window(load("positions"))
stop_events = in_window(load("stop_events"))
scheduled_trips = in_window(load("scheduled_trips"))
predictions = load("predictions")
predictions = in_window(predictions.assign(start_date=predictions["trip_key"].str.split("|").str[1]))
frames = load("frames")
frames = frames[frames["service_date"].between(date(2026, 10, 16), date(2026, 10, 28))]

KEY = ["trip_key", "vehicle_id", "stop_sequence"]  # one stop visit by one bus

## 1. Trips scheduled vs observed

Per service day: scheduled trips (511NY GTFS) split into those seen in the feed with a vehicle and those never seen. A trip that isn't seen either didn't run or ran without tracking. From the rider's side both mean no live info for that trip.

In [ ]:
observed_ids = positions[["start_date", "trip_id"]].drop_duplicates()
sched = scheduled_trips[["start_date", "trip_id"]].merge(observed_ids, how="left", indicator=True)
sched["observed"] = sched["_merge"] == "both"
extra = observed_ids.merge(scheduled_trips[["start_date", "trip_id"]], how="left", indicator=True)
print("observed trips not in that day's schedule:", (extra["_merge"] == "left_only").sum())

daily = sched.groupby("start_date")["observed"].agg(scheduled="size", observed="sum")
daily["not_observed"] = daily["scheduled"] - daily["observed"]
daily["pct_observed"] = daily["observed"] / daily["scheduled"]

days = pd.to_datetime(daily.index)
x = np.arange(len(daily))
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.bar(x, daily["observed"], color=GREEN, label="Observed with a live vehicle")
ax.bar(x, daily["not_observed"], bottom=daily["observed"], color=GRAY,
       label="Scheduled, not observed (not run or not tracked)")
for xi, (obs, tot) in enumerate(zip(daily["observed"], daily["scheduled"])):
    ax.text(xi, tot + 4, f"{obs / tot:.0%}", ha="center", va="bottom", fontsize=9, color="#333333")
ax.set_xticks(x, [f"{d:%a}\n{d:%b} {d.day}" for d in days])
ax.set_ylabel("Trips")
ax.set_ylim(0, daily["scheduled"].max() * 1.1)
ax.grid(axis="x", visible=False)
ax.set_title("Hudson Link Trips Scheduled vs Observed, Oct 16 to 28 2026")
fig.legend(loc="lower center", bbox_to_anchor=(0.5, 0.04), ncol=2, frameon=False)
fig.subplots_adjust(bottom=0.24)
add_source_footnote(fig, SOURCE)
fig.savefig(CHARTS / "trips_scheduled_vs_observed.png", bbox_inches="tight")
plt.show()

tot = daily[["scheduled", "observed"]].sum()
print(f"Window total: {tot['observed']:,} of {tot['scheduled']:,} scheduled trips observed "
      f"({tot['observed'] / tot['scheduled']:.0%})")
daily

## 2. ETA completeness: did the feed cover the scheduled trip-stops?

This follows the [Swiftly ETA Completeness Benchmark](https://github.com/SwiftlyInc/ETA-Completeness-Benchmark), which asks only whether a prediction was there, not whether it was right:

> **ETA Completeness Score** = complete trip-stops / all scheduled trip-stops

The unit is a **scheduled trip-stop**: one scheduled trip at one stop on one service day (every GTFS `stop_times` row for each scheduled trip, so trips that never appeared in the feed count against the score). A trip-stop is **complete** if either:
1. the trip had an assigned vehicle **and** at least one prediction between 0 and 15 minutes out for that stop, or
2. the trip was marked `CANCELED`, or the stop `SKIPPED`, in the GTFS-rt trip updates.

Anything else is incomplete. The benchmark's tiers: **95%+ best in class**, 90–94% good, 85–89% fair, under 85% needs improvement.

How the benchmark is applied here:
- **Assigned vehicle:** the trip appears in `positions` with a vehicle on that service day.
- **0–15 minutes out:** the feed's own predicted time to arrival (`source_pred_t` minus the poll time) was between 0 and 15 min in at least one published poll. It does not use the actual arrival, so it works for trips that never ran. Predictions already in the past are ignored.
- **CANCELED / SKIPPED:** this recorded feed never publishes either, so condition 2 is never met. A trip that did not run shows up as having no vehicle.
- The benchmark leaves out service alerts and trip modifications, and so does this.

The PNG is a one-page scorecard of the numbers.

In [ ]:
stop_times = pd.read_csv("data/raw/gtfs_schedule/stop_times.txt", dtype=str, usecols=["trip_id", "stop_sequence"])
stop_times["stop_sequence"] = stop_times["stop_sequence"].astype(int)

# denominator: every scheduled trip-stop in the window
trip_stops = (scheduled_trips[["trip_id", "start_date", "route_id"]]
              .merge(stop_times, on="trip_id"))
TS_KEY = ["trip_id", "start_date", "stop_sequence"]

# assigned vehicle: the trip appears in the feed with a vehicle that day
assigned = positions.loc[positions["vehicle_id"].notna(), ["trip_id", "start_date"]].drop_duplicates()
trip_stops = trip_stops.merge(assigned.assign(has_vehicle=True), how="left", on=["trip_id", "start_date"])
trip_stops["has_vehicle"] = trip_stops["has_vehicle"].fillna(False).astype(bool)

# at least one published prediction 0-15 min out for that stop
preds = predictions.assign(trip_id=predictions["trip_key"].str.split("|").str[0])
eta_min = (preds["source_pred_t"] - preds["frame_t"]) / 60
in_window_15 = preds.loc[eta_min.between(0, 15), TS_KEY].drop_duplicates().assign(has_pred_0_15=True)
trip_stops = trip_stops.merge(in_window_15, how="left", on=TS_KEY)
trip_stops["has_pred_0_15"] = trip_stops["has_pred_0_15"].fillna(False).astype(bool)

# CANCELED / SKIPPED are never published in this feed
trip_stops["canceled_or_skipped"] = False
trip_stops["complete"] = (trip_stops["has_vehicle"] & trip_stops["has_pred_0_15"]) | trip_stops["canceled_or_skipped"]
trip_stops["reason"] = np.select(
    [trip_stops["complete"], ~trip_stops["has_vehicle"]],
    ["complete", "no_vehicle"],
    default="no_pred_0_15",
)

TIERS = [(0.95, "Best in class", "#3D9970"), (0.90, "Good", "#7BB662"),
         (0.85, "Fair", "#F4A259"), (0.0, "Needs improvement", "#D1495B")]


def tier(score):
    return next((name, color) for cut, name, color in TIERS if score >= cut)


n_all = len(trip_stops)
score = trip_stops["complete"].mean()
reasons = trip_stops["reason"].value_counts()
by_day = trip_stops.groupby("start_date")["complete"].agg(["mean", "sum", "size"])
by_route = trip_stops.groupby("route_id")["complete"].agg(["mean", "sum", "size"]).sort_index()
print(f"ETA Completeness Score: {score:.1%} ({trip_stops['complete'].sum():,} of {n_all:,} scheduled trip-stops), "
      f"{tier(score)[0]}")

# --- scorecard PNG
fig = plt.figure(figsize=(12, 7.5))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.15], hspace=0.55, wspace=0.35,
                      left=0.05, right=0.97, top=0.88, bottom=0.1)
fig.suptitle("Hudson Link ETA Completeness Score, Oct 16 to 28 2026", fontsize=15, fontweight="bold", x=0.05, ha="left")

# headline
ax = fig.add_subplot(gs[0, 0]); ax.axis("off")
name, color = tier(score)
ax.text(0, 0.78, f"{score:.1%}", fontsize=48, fontweight="bold", color=color, va="center")
ax.text(0, 0.42, name, fontsize=16, fontweight="bold", color=color, va="center")
ax.text(0, 0.18, f"{trip_stops['complete'].sum():,} complete of {n_all:,}\nscheduled trip-stops", fontsize=11, color="#333333", va="center")

# outcome breakdown
ax = fig.add_subplot(gs[0, 1:]); ax.axis("off")
ax.set_title("Where the scheduled trip-stops went", loc="left", fontsize=11)
rows = [("Complete: vehicle assigned and a prediction 0 to 15 min out", reasons.get("complete", 0), GREEN),
        ("Incomplete: vehicle assigned, no prediction 0 to 15 min out", reasons.get("no_pred_0_15", 0), "#F4A259"),
        ("Incomplete: no vehicle assigned to the trip", reasons.get("no_vehicle", 0), "#D1495B")]
for i, (label, cnt, col) in enumerate(rows):
    y = 0.80 - i * 0.30
    ax.add_patch(plt.Rectangle((0, y - 0.07), 0.62 * cnt / n_all, 0.14, color=col, transform=ax.transAxes, clip_on=False))
    ax.text(0.0, y + 0.13, label, fontsize=9, color="#333333", transform=ax.transAxes, va="center")
    ax.text(0.62 * cnt / n_all + 0.01, y, f"{cnt:,} ({cnt / n_all:.1%})", fontsize=10, fontweight="bold",
            transform=ax.transAxes, va="center")

# by day
ax = fig.add_subplot(gs[1, :2])
d = pd.to_datetime(by_day.index)
ax.bar(range(len(by_day)), by_day["mean"], color=[tier(v)[1] for v in by_day["mean"]])
for xi, v in enumerate(by_day["mean"]):
    ax.text(xi, v + 0.01, f"{v:.0%}", ha="center", fontsize=8)
for cut, _, _ in TIERS[:3]:
    ax.axhline(cut, color="#666666", lw=0.8, ls="--")
ax.set_xticks(range(len(by_day)), [f"{x:%a}\n{x:%b} {x.day}" for x in d], fontsize=8)
ax.set_ylim(0, 1.1); ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.grid(axis="x", visible=False)
ax.set_title("% of trip-stops complete, by service day (dashed: 85%, 90%, 95% tiers)", loc="left", fontsize=11)

# by route
ax = fig.add_subplot(gs[1, 2])
ax.barh(by_route.index, by_route["mean"], color=[tier(v)[1] for v in by_route["mean"]])
for yi, v in enumerate(by_route["mean"]):
    ax.text(v + 0.01, yi, f"{v:.0%}", va="center", fontsize=8)
ax.invert_yaxis(); ax.set_xlim(0, 1.15)
ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.grid(axis="y", visible=False)
ax.set_title("% of trip-stops complete, by route", loc="left", fontsize=11)

add_source_footnote(fig, SOURCE + "; method: Swiftly ETA Completeness Benchmark")
fig.text(0.99, 0.035, "Note: this feed has no CANCELED or SKIPPED status, so every complete trip-stop here is a vehicle plus a prediction.",
         ha="right", va="bottom", fontsize=8, color="#666666")
fig.savefig(CHARTS / "eta_completeness_scorecard.png", bbox_inches="tight")
plt.show()

by_route.rename(columns={"mean": "score", "sum": "complete", "size": "scheduled"}).assign(
    score=lambda t: t["score"].map("{:.1%}".format))

## 3. Prediction accuracy: source (CoachUSA) vs actual

Every published prediction for a stop the bus later reached, pooled over the window. The x-axis is time before arrival, running left to right toward the bus arriving at 0: how far ahead the prediction was made, i.e. the actual arrival minus the time it was published. The y-axis is **actual − predicted**: above zero means the bus came later than shown, below zero means earlier. Earlier is worse for riders, since they can miss the bus. The line is the median, the bands are the middle 50% and 80% of predictions. "Actual" is inferred from ~1-minute GPS, so treat errors under about a minute as noise.

In [ ]:
acc = predictions.dropna(subset=["actual_t"]).query("0 <= horizon_s < 3600").copy()
acc["horizon_min"] = (acc["horizon_s"] // 60).astype(int)
acc["late_min"] = -acc["source_err_s"] / 60  # actual - predicted
q = acc.groupby("horizon_min")["late_min"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).unstack()

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.fill_between(q.index, q[0.1], q[0.9], color=BLUE, alpha=0.15, lw=0, label="Middle 80%")
ax.fill_between(q.index, q[0.25], q[0.75], color=BLUE, alpha=0.3, lw=0, label="Middle 50%")
ax.plot(q.index, q[0.5], color=BLUE, lw=2, label="Median")
ax.axhline(0, color="#333333", lw=1)
ax.text(0.5, 0.97, "Bus later than predicted", transform=ax.transAxes, ha="center", va="top", fontsize=9, color="#666666")
ax.text(0.5, 0.03, "Bus earlier than predicted", transform=ax.transAxes, ha="center", va="bottom", fontsize=9, color="#666666")
ax.set_xlim(59, 0)  # time runs toward arrival at the right
ax.set_xlabel("Minutes before the bus arrived")
ax.set_ylabel("Actual minus predicted arrival (min)")
ax.set_title("Hudson Link Prediction Error by Time to Arrival, Oct 16 to 28 2026")
fig.legend(loc="lower center", bbox_to_anchor=(0.5, 0.04), ncol=3, frameon=False)
fig.subplots_adjust(bottom=0.24)
add_source_footnote(fig, SOURCE)
fig.savefig(CHARTS / "prediction_error_by_horizon.png", bbox_inches="tight")
plt.show()

acc["bucket"] = pd.cut(acc["horizon_s"] / 60, [0, 5, 15, 30, 60], right=False,
                       labels=["0-5 min", "5-15 min", "15-30 min", "30-60 min"])
acc["abs_min"] = acc["late_min"].abs()
acc.groupby("bucket", observed=True).agg(
    predictions=("late_min", "size"),
    mae_min=("abs_min", "mean"),
    median_min=("late_min", "median"),
    p90_abs_min=("abs_min", lambda s: s.quantile(0.9)),
    within_2_min=("abs_min", lambda s: (s <= 2).mean()),
    bus_2plus_min_early=("late_min", lambda s: (s < -2).mean()),
).round(2).style.format({"predictions": "{:,}", "within_2_min": "{:.0%}", "bus_2plus_min_early": "{:.0%}"},
                        precision=2)

## Next metrics

- **Fleet size:** distinct vehicles in service per 15 min, peak vs midday vs weekend, and interlining across routes (`positions`).
- **Frequency:** scheduled vs delivered headways at one key stop per route/direction, using GTFS `stop_times` vs `stop_events.actual_dep_time`. First pick the timepoints.
- **On-time performance:** origin departure delay and terminal arrival delay from `stop_events`, within a −1/+5 min band, by route and hour. Also where along the route delay builds up. Use `actual_approx == False` for the cleanest numbers.
- **Prediction issues in depth:** `flags` (`jump` / `behind` / `diverge`) by route and stop, prediction "wobble" between polls, and accuracy by route and time of day.
- **Missed vs untracked:** check the gray trips in section 1 against feed-outage windows (`frames.err`) and `n_unresolved`.